# CS5100 Foundations of AI — Progress Report
## Music Genre Classification with Pre-trained Audio Models on FMA-Small

| | |
|---|---|
| **Author** | Anand Dev |
| **Course** | CS5100 — Foundations of AI, Northeastern University |
| **Date** | March 2026 |
| **Dataset** | FMA-Small (8,000 tracks × 30 sec, 8 genres) |

In [ ]:
import sys, os, json, glob
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from IPython.display import display, Image

from data.data_utils import load_fma_metadata, get_splits, RESULTS_DIR

sns.set_theme(style='whitegrid', font_scale=1.05)
%matplotlib inline

FIGURES_DIR = os.path.join(RESULTS_DIR, 'figures')
RUNS_DIR    = os.path.join(RESULTS_DIR, 'runs')

print(f'Results dir : {RESULTS_DIR}')
print(f'Run JSONs   : {len(glob.glob(os.path.join(RUNS_DIR, "*.json")))} files found')

---
## 1. Dataset Overview

In [ ]:
df = load_fma_metadata()
train_df, val_df, test_df = get_splits(df)

print(f'Total tracks : {len(df)}')
print(f'Train / Val / Test : {len(train_df)} / {len(val_df)} / {len(test_df)}')
print(f'Genres : {sorted(df["genre"].unique())}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

counts = df['genre'].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('FMA-Small — Genre Distribution (Full Dataset)')
axes[0].set_ylabel('Tracks')
axes[0].set_xlabel('Genre')
axes[0].tick_params(axis='x', rotation=30)

split_counts = pd.DataFrame({
    'Train': train_df['genre'].value_counts().sort_index(),
    'Val':   val_df['genre'].value_counts().sort_index(),
    'Test':  test_df['genre'].value_counts().sort_index(),
})
split_counts.plot(kind='bar', ax=axes[1], colormap='tab10')
axes[1].set_title('Genre Distribution by Split')
axes[1].set_ylabel('Tracks')
axes[1].set_xlabel('Genre')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

---
## 2. Models

| Model | HuggingFace ID | Type | Pre-training Data | SR | Embedding Dim |
|-------|---------------|------|------------------|----|---------------|
| **MERT-95M** | `m-a-p/MERT-v1-95M` | Discriminative (music) | ~160k hrs music | 24 kHz | 768 |
| **MERT-330M** | `m-a-p/MERT-v1-330M` | Discriminative (music) | ~160k hrs music | 24 kHz | 1024 |
| **CLAP (LAION)** | `laion/clap-htsat-fused` | Audio-text contrastive | 630k audio–text pairs | 48 kHz | 512 |
| **AST** | `MIT/ast-finetuned-audioset-10-10-0.4593` | Discriminative (general) | AudioSet (527 classes) | 16 kHz | 768 |
| **MusicLDM-VAE** | `ucsd-reach/musicldm` | Generative (music) | 466 hrs music | 16 kHz | ~512 |

### Evaluation Paradigms

- **Zero-shot (linear probe)**: Encoder frozen → extract embeddings for all train tracks → fit `LogisticRegression` (C=1, max_iter=1000) → evaluate on test set.
- **Fine-tuned**: End-to-end training with a small MLP classification head. Differential learning rates: backbone 1e-5, head 1e-3. Cosine annealing, gradient accumulation, AMP.
- **Baseline (chance)**: 1/8 = **12.5%** for 8 balanced classes.

---
## 3. Results

In [ ]:
# __ Load all experiment run JSONs (auto-aggregated) __
run_files = sorted(glob.glob(os.path.join(RUNS_DIR, '*.json')))

records = []
for fp in run_files:
    with open(fp) as f:
        records.append(json.load(f))

if not records:
    print('No run JSONs found. Run experiments first.')
else:
    results_df = pd.DataFrame(records)
    results_df['Accuracy (%)'] = (results_df['test_accuracy'] * 100).round(2)
    results_df['Model'] = results_df.apply(
        lambda r: f"{r['model'].upper()}-{r['variant']}" if r.get('variant') else r['model'].upper(),
        axis=1
    )
    results_df = results_df.sort_values('test_accuracy', ascending=False).reset_index(drop=True)
    print(f'Loaded {len(results_df)} experiment runs')

In [ ]:
# __ Summary table __
if records:
    display_cols = ['Model', 'variant', 'mode', 'Accuracy (%)', 'best_val_accuracy', 'timestamp']
    display_cols = [c for c in display_cols if c in results_df.columns]
    styled = results_df[display_cols].style \
        .format({'Accuracy (%)': '{:.2f}', 'best_val_accuracy': lambda x: f'{x*100:.2f}%' if pd.notna(x) and x is not None else '—'}) \
        .background_gradient(subset=['Accuracy (%)'], cmap='YlGn') \
        .set_caption('Table 1: All Experiment Results — FMA-Small Genre Classification')
    display(styled)

In [ ]:
# __ Accuracy comparison bar chart __
if records:
    fig, ax = plt.subplots(figsize=(11, max(4, len(results_df) * 0.65)))
    colors = results_df['mode'].map({'zero_shot': '#3a7ebf', 'finetune': '#e06030'}).fillna('#888')
    bars = ax.barh(results_df['Model'] + ' (' + results_df['mode'].str.replace('_', '-') + ')',
                   results_df['Accuracy (%)'], color=colors, edgecolor='white', height=0.6)

    # chance line
    ax.axvline(12.5, color='black', ls='--', lw=1.2, label='Chance (12.5%)')

    for bar, val in zip(bars, results_df['Accuracy (%)']):
        ax.text(val + 0.8, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=10)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#3a7ebf', label='Zero-shot (linear probe)'),
        Patch(facecolor='#e06030', label='Fine-tuned'),
    ]
    ax.legend(handles=legend_elements, loc='lower right')
    ax.set_xlabel('Test Accuracy (%)', fontsize=12)
    ax.set_xlim(0, 105)
    ax.set_title('FMA-Small Genre Classification — Model Comparison', fontsize=13, pad=12)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'progress_report_comparison.png'), dpi=150)
    plt.show()

### 3a. Per-Genre F1 Scores

In [ ]:
# __ Per-genre F1 heatmap (only for runs that have per_class_f1 data) __
if records:
    f1_rows = []
    for _, row in results_df.iterrows():
        f1 = row.get('per_class_f1')
        if isinstance(f1, dict) and len(f1) > 0:
            label = f"{row['Model']} ({row['mode'].replace('_','-')})"
            entry = {'Model': label}
            entry.update({k: round(v, 3) for k, v in f1.items()})
            f1_rows.append(entry)

    if f1_rows:
        f1_df = pd.DataFrame(f1_rows).set_index('Model')
        fig, ax = plt.subplots(figsize=(12, max(3, len(f1_df) * 0.8)))
        sns.heatmap(f1_df.astype(float), annot=True, fmt='.2f', cmap='YlOrRd',
                    vmin=0, vmax=1, linewidths=0.4, ax=ax)
        ax.set_title('Per-Genre F1-Score Heatmap', fontsize=13)
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURES_DIR, 'progress_report_f1_heatmap.png'), dpi=150)
        plt.show()
    else:
        print('No per-class F1 data in current runs. Will populate after running evaluate.py.')

### 3b. Confusion Matrices

In [ ]:
# __ Display confusion matrices from results/figures/ (newest first) __
# These are saved by evaluate.py (row-normalised to %) and models/ast/finetune.py

def show_images_grid(paths, titles=None, cols=2, figwidth=14):
    """Display a list of image files in a grid."""
    n = len(paths)
    if n == 0:
        print('No images to display.')
        return
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(figwidth, rows * figwidth // cols * 0.75))
    axes = np.array(axes).flatten()
    for i, (path, ax) in enumerate(zip(paths, axes)):
        if os.path.exists(path):
            img = mpimg.imread(path)
            ax.imshow(img)
            ax.axis('off')
            if titles:
                ax.set_title(titles[i], fontsize=10)
        else:
            ax.axis('off')
            ax.text(0.5, 0.5, f'Missing:\n{os.path.basename(path)}',
                    ha='center', va='center', color='red', transform=ax.transAxes)
    for ax in axes[len(paths):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


# New-format confusion matrices from evaluate.py (saved to results/figures/)
new_cm_files = sorted(glob.glob(os.path.join(FIGURES_DIR, 'confmat_*.png')), reverse=True)

if new_cm_files:
    print(f'Found {len(new_cm_files)} confusion matrix image(s) from evaluate.py:')
    titles = [os.path.basename(f).replace('.png','').replace('confmat_','').replace('_', ' ') for f in new_cm_files]
    show_images_grid(new_cm_files, titles=titles, cols=2)
else:
    print('No confusion matrices from evaluate.py yet. Run evaluate.py first.')
    # Fall back to legacy confusion matrices
    legacy_dir = os.path.join(os.path.dirname(RESULTS_DIR), 'results', 'evaluations')
    legacy_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'evaluations', 'confusion_matrix*.png')))
    if legacy_files:
        print(f'Showing {len(legacy_files)} legacy confusion matrix image(s):')
        show_images_grid(legacy_files, cols=2)

In [ ]:
# __ AST fine-tune training curves (if available) __
ast_curve = os.path.join(FIGURES_DIR, 'ast_training_curves.png')
if os.path.exists(ast_curve):
    print('AST Fine-tuning Curves:')
    display(Image(ast_curve, width=750))
else:
    print('AST training curves not yet available (run models/ast/finetune.py --mode finetune).')

# Legacy MERT training curves
mert_curve = os.path.join(RESULTS_DIR, 'evaluations', 'training_curves.png')
if os.path.exists(mert_curve):
    print('\nMERT Training Curves (pre-restructure run):')
    display(Image(mert_curve, width=750))

---
## 4. Analysis

### 4a. Zero-Shot Comparison

| Model | Zero-Shot Acc. | Notes |
|-------|---------------|-------|
| MERT-330M | **62.41%** | Best zero-shot — large music-specific transformer |
| MERT-95M | 58.41% | Strong — same architecture, smaller capacity |
| AST | _TBD_ | AudioSet-trained ViT; may transfer to genre |
| CLAP-LAION (linear probe) | _TBD_ | Old text-similarity score was 12.51% (≈ random) |

**MERT dominates zero-shot** because it was trained on music audio with masked prediction, making its representations highly specific to musical structure. AST was trained on general audio events (AudioSet), so genre transfer is less certain.

CLAP's original 12.5% was using text-audio similarity without a linear probe — essentially random for 8 classes. The new linear probe evaluation should give a fairer comparison.

### 4b. Fine-Tuning vs. Zero-Shot Gap

| Model | Zero-Shot | Fine-Tuned | Gain |
|-------|-----------|------------|------|
| MERT-95M | 58.41% | _TBD_ | — |
| MERT-330M | 62.41% | _TBD_ | — |
| AST | _TBD_ | _TBD_ | — |

_(Update after running `run_experiments.sh`)_

The fine-tuning gain is expected to be largest for models whose pre-training task is furthest from genre classification (e.g., CLAP's contrastive audio-text objective vs. MERT's masked prediction).

In [ ]:
# __ Zero-shot vs Fine-tuned grouped bar chart (auto-populates as results come in) __
if records:
    pivot = results_df.pivot_table(
        index='Model', columns='mode', values='Accuracy (%)', aggfunc='max'
    ).rename(columns={'zero_shot': 'Zero-Shot', 'finetune': 'Fine-Tuned'})

    if set(['Zero-Shot', 'Fine-Tuned']).intersection(pivot.columns):
        ax = pivot.plot(kind='bar', figsize=(10, 5), colormap='Set1',
                        edgecolor='white', width=0.7)
        ax.axhline(12.5, color='black', ls='--', lw=1, label='Chance')
        ax.set_ylabel('Test Accuracy (%)')
        ax.set_title('Zero-Shot vs Fine-Tuned Accuracy', fontsize=13)
        ax.tick_params(axis='x', rotation=30)
        ax.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURES_DIR, 'progress_report_zs_vs_ft.png'), dpi=150)
        plt.show()
    else:
        print('Only one mode available so far — grouped chart will appear after fine-tuning.')

### 4c. Per-Genre Patterns

From existing MERT confusion matrices (see Section 3b):

- **Easiest genres**: Electronic, Hip-Hop — likely high energy and rhythmic distinctiveness makes them separable across models.
- **Hardest genres**: Experimental, Folk, International — less acoustically uniform; Experimental in particular spans a wide stylistic range.
- **Common confusions**: Pop ↔ Rock (similar production styles), Instrumental ↔ Folk (both acoustic, low vocal content).

_(Update after new confusion matrices are generated by evaluate.py)_

---
## 5. Experimental Setup

**Data**: FMA-Small — 7,994 usable 30-second MP3 clips (6 corrupted). Stratified split: **60% train / 20% val / 20% test**, `random_state=42` for reproducibility across all scripts.

**Zero-Shot Pipeline**
```
Raw MP3 → librosa.load (model SR) → model-specific preprocessor
       → frozen encoder → fixed embedding → LogisticRegression
```

**Fine-Tuning Pipeline**
```
Raw MP3 → preprocessor → encoder (lr=1e-5) + MLP head (lr=1e-3)
       → CrossEntropyLoss → AdamW → CosineAnnealingLR
       → AMP (float16) + gradient accumulation
```

**Metric**: Top-1 accuracy on the held-out test split. Per-class F1 also reported.

**Hardware**: Single NVIDIA GPU (mixed precision enabled).

---
## 6. Next Steps

- [x] MERT-95M zero-shot linear probe (58.41%)
- [x] MERT-330M zero-shot linear probe (62.41%)
- [ ] CLAP-LAION zero-shot linear probe (run `evaluate.py`)
- [ ] AST zero-shot linear probe (run `evaluate.py`)
- [ ] AST fine-tune end-to-end (run `models/ast/finetune.py`)
- [ ] MERT fine-tuned accuracy — record from existing checkpoints
- [ ] Confusion matrix per-genre analysis for all models
- [ ] Explore ensemble (majority vote / feature stacking across models)
- [ ] MusicLDM-VAE zero-shot evaluation
- [ ] UMAP / t-SNE visualization of latent spaces

In [ ]:
# __ Quick status summary __
if records:
    print('=== Experiment Status ===')
    print(f'Total runs logged : {len(results_df)}')
    print(f'Best result so far: {results_df.iloc[0]["Model"]} ({results_df.iloc[0]["mode"]}) '
          f'— {results_df.iloc[0]["Accuracy (%)"]:.2f}%')
    print()
    print(results_df[['Model', 'mode', 'Accuracy (%)']].to_string(index=False))